In [1]:
import pandas as pd
import numpy as np
import os

print("Starting Form & Dominance Analysis...")

# 1. Load Data and Standardize Teams
matches = pd.read_csv('../data/raw/matches.csv')

team_mapping = {
    'Delhi Daredevils': 'Delhi Capitals', 'Kings XI Punjab': 'Punjab Kings', 
    'Deccan Chargers': 'Sunrisers Hyderabad', 'Rising Pune Supergiants': 'Rising Pune Supergiant',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru', 'Gujarat Lions': 'Gujarat Titans' # Grouping new franchises if desired
}
matches['team1'] = matches['team1'].replace(team_mapping)
matches['team2'] = matches['team2'].replace(team_mapping)
matches['winner'] = matches['winner'].replace(team_mapping)

matches['date'] = pd.to_datetime(matches['date'])
matches = matches.sort_values('date').reset_index(drop=True)

# ---------------------------------------------------------
# PART A: Rolling 5-Match Form
# ---------------------------------------------------------
# Create a single column of all matches played by each team
team1_df = matches[['id', 'date', 'team1', 'winner']].rename(columns={'team1': 'team'})
team2_df = matches[['id', 'date', 'team2', 'winner']].rename(columns={'team2': 'team'})
all_team_matches = pd.concat([team1_df, team2_df]).sort_values(by=['team', 'date']).reset_index(drop=True)

# 1 if they won, 0 if they lost or tied/no result
all_team_matches['won'] = (all_team_matches['team'] == all_team_matches['winner']).astype(int)

# Calculate win percentage over the last 5 matches (Rolling Mean)
all_team_matches['rolling_5_form'] = all_team_matches.groupby('team')['won'].transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())

# Fill NaNs for the very first match a team plays
all_team_matches['rolling_5_form'].fillna(0.5, inplace=True) 

# ---------------------------------------------------------
# PART B: Dominance Score (Head-to-Head)
# ---------------------------------------------------------
# Create a unique matchup string (e.g., 'Chennai Super Kings vs Mumbai Indians')
# Sorting ensures 'MI vs CSK' and 'CSK vs MI' are treated identically
matches['matchup'] = matches.apply(lambda x: ' vs '.join(sorted([str(x['team1']), str(x['team2'])])), axis=1)

# Count total matches and wins per matchup
h2h_stats = matches.groupby(['matchup', 'winner']).size().reset_index(name='wins')
h2h_totals = matches.groupby('matchup').size().reset_index(name='total_matches')

# Merge to get win ratio
dominance_df = pd.merge(h2h_stats, h2h_totals, on='matchup')
dominance_df['dominance_score'] = (dominance_df['wins'] / dominance_df['total_matches']).round(2)

# ---------------------------------------------------------
# PART C: Save the Features
# ---------------------------------------------------------
os.makedirs('../data/processed', exist_ok=True)
all_team_matches[['id', 'team', 'rolling_5_form']].to_csv('../data/processed/team_form.csv', index=False)
dominance_df.to_csv('../data/processed/dominance_matrix.csv', index=False)

print("✅ Form & Dominance scores engineered successfully!")
print("\nSample Dominance Matrix (Head-to-Head Win %):")
display(dominance_df.sort_values(by='total_matches', ascending=False).head(10))

Starting Form & Dominance Analysis...
✅ Form & Dominance scores engineered successfully!

Sample Dominance Matrix (Head-to-Head Win %):


C:\Users\garvi\AppData\Local\Temp\ipykernel_16316\1114187548.py:37: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  all_team_matches['rolling_5_form'].fillna(0.5, inplace=True)


,matchup,winner,wins,total_matches,dominance_score
83,Kolkata Knight Riders vs Sunrisers Hyderabad,Sunrisers Hyderabad,11,37,0.30
11,Chennai Super Kings vs Mumbai Indians,Mumbai Indians,20,37,0.54
10,Chennai Super Kings vs Mumbai Indians,Chennai Super Kings,17,37,0.46
82,Kolkata Knight Riders vs Sunrisers Hyderabad,Kolkata Knight Riders,26,37,0.70
129,Royal Challengers Bengaluru vs Sunrisers Hyder...,Royal Challengers Bengaluru,16,35,0.46
30,Delhi Capitals vs Mumbai Indians,Delhi Capitals,16,35,0.46
130,Royal Challengers Bengaluru vs Sunrisers Hyder...,Sunrisers Hyderabad,19,35,0.54
43,Delhi Capitals vs Sunrisers Hyderabad,Sunrisers Hyderabad,17,35,0.49
42,Delhi Capitals vs Sunrisers Hyderabad,Delhi Capitals,18,35,0.51
31,Delhi Capitals vs Mumbai Indians,Mumbai Indians,19,35,0.54
